# Juja terrain and 3D building showcase

This notebook is the optional Juja-only terrain visualization output. The operational dashboard stays 2D. Start the FastAPI backend first so the local DEM terrain-RGB tiles are available.

Set `MAPTILER_KEY` in the notebook environment before running the map cell. The key is intentionally read from the environment and is not stored in this notebook.

In [ ]:
%pip install "leafmap[maplibre]"

In [ ]:
import leafmap.maplibregl as leafmap

API = "http://localhost:8000/api/v1"
CENTER = [37.0111, -1.1027]
DEM_TILES = f"{API}/terrain/dem/{{z}}/{{x}}/{{y}}.png"
MAPTILER_KEY = leafmap.get_api_key("MAPTILER_KEY")
if not MAPTILER_KEY:
    raise RuntimeError("Set MAPTILER_KEY in the notebook environment before running this cell.")

style = {
    "version": 8,
    "sources": {
        "osm": {
            "type": "raster",
            "tiles": ["https://a.tile.openstreetmap.org/{z}/{x}/{y}.png"],
            "tileSize": 256,
            "attribution": "&copy; OpenStreetMap Contributors",
            "maxzoom": 19,
        },
        "terrainSource": {
            "type": "raster-dem",
            "tiles": [DEM_TILES],
            "tileSize": 256,
            "encoding": "mapbox",
            "maxzoom": 14,
        },
        "hillshadeSource": {
            "type": "raster-dem",
            "tiles": [DEM_TILES],
            "tileSize": 256,
            "encoding": "mapbox",
            "maxzoom": 14,
        },
    },
    "layers": [
        {"id": "osm", "type": "raster", "source": "osm"},
        {
            "id": "juja-hillshade",
            "type": "hillshade",
            "source": "hillshadeSource",
            "layout": {"visibility": "visible"},
            "paint": {
                "hillshade-shadow-color": "#473B24",
                "hillshade-highlight-color": "#fff7d6",
                "hillshade-illumination-direction": 315,
            },
        },
    ],
    "terrain": {"source": "terrainSource", "exaggeration": 1.6},
}

m = leafmap.Map(center=CENTER, zoom=13, pitch=55, bearing=145, style=style)
m.add_basemap("Esri.WorldImagery", visible=False)

source = {
    "url": f"https://api.maptiler.com/tiles/v3/tiles.json?key={MAPTILER_KEY}",
    "type": "vector",
}

layer = {
    "id": "3d-buildings",
    "source": "openmaptiles",
    "source-layer": "building",
    "type": "fill-extrusion",
    "minzoom": 15,
    "paint": {
        "fill-extrusion-color": [
            "case",
            ["==", ["get", "class"], "unclassified"], "#7f9497",
            "interpolate",
            ["linear"],
            ["coalesce", ["get", "render_height"], 6],
            0,
            "#6d8791",
            20,
            "#d5a34b",
            70,
            "#c86b48",
        ],
        "fill-extrusion-height": [
            "interpolate",
            ["linear"],
            ["zoom"],
            15,
            0,
            16,
            ["coalesce", ["get", "render_height"], 6],
        ],
        "fill-extrusion-base": [
            "case",
            [">=", ["zoom"], 16],
            ["coalesce", ["get", "render_min_height"], 0],
            0,
        ],
        "fill-extrusion-opacity": 0.9,
    },
}

m.add_source("openmaptiles", source)
m.add_layer(layer)
m.add_layer_control(bg_layers=True)
m

## Optional QGIS hillshade output

The native MapLibre hillshade above is generated from the conditioned DEM. The QGIS `hillshade.tif` remains available from `GET /api/v1/terrain/hillshade.png` when a static raster output is needed. Keep it as a separate output so it is not confused with the 3D terrain surface.